# Ribbon graph generation

This tutorial constructs trivalent (i.e., three edges for every vertex) ribbon graphs of genus $g$ with $F$ faces. A ribbon graph consists of a graph (a collection of vertices and edges) together with a cyclic ordering of the incident edges at every vertex. The cyclic order determines the oriented surface obtained by thickening the graph; its faces correspond to punctures of the associated Riemann surface.

For a connected trivalent ribbon graph, Euler's relation and trivalence give

$$
E=3(2g-2+F), \qquad V=2(2g-2+F).
$$

Here, we generate all the graphs for genus 1 and 2, and then reproducibly sample one ribbon graph at genus 4.

In [1]:
from pprint import pprint

# there are three different submodules in the string_amplitudes package
from string_amplitudes import (
    generate_ribbon_graphs,
    get_boundary_data,
    sample_ribbon_graph,
)

## Ribbon graph generation at genus 1

With one face, genus 1 requires three edges and two trivalent vertices. There is one unique ribbon graph.

In [2]:
# specifying the number of faces (vertex operators) and genus is all the information necessary
# to generate the ribbon graphs.
genus = 1
n_faces = 1
graphs_g1 = generate_ribbon_graphs(genus=genus, n_faces=n_faces)
print(f"Number of non-isomorphic ribbon graphs: {len(graphs_g1)}")


# Let's inspect the unique (up to isomorphism) graph
graph = graphs_g1[0]
# The ribbon graph structure is a tuple of the edge labels, vertex labels,
# and the cyclic ordering of edges at each vertex, which we call the "rotation system".
edges, vertices, rotation = graph
print("Edges:")
pprint(edges)
print("Vertices:")
pprint(vertices)
print("Cyclic edge order at each vertex:")
pprint(rotation)

Number of non-isomorphic ribbon graphs: 1
Edges:
[(0, 1), (0, 1), (0, 1)]
Vertices:
[0, 1]
Cyclic edge order at each vertex:
{0: [0, 1, 2], 1: [0, 1, 2]}


Each edge is stored by the labels of its two endpoints. Parallel edges can occur, so the same endpoint pair can occur more than once. The dictionary `rotation` stores edge indices. `rotation[v]` is the cyclic order of the three edges incident on vertex `v`.

We can verify the required numbers of vertices and edges directly.

In [3]:
expected_edges = 3 * (2 * genus - 2 + n_faces)
expected_vertices = 2 * (2 * genus - 2 + n_faces)
assert len(edges) == expected_edges
assert len(vertices) == expected_vertices
print(f"V = {len(vertices)}, E = {len(edges)}, F = {n_faces}")

V = 2, E = 3, F = 1


## Face boundary and sewing data

`get_boundary_data` traverses the edges of the ribbon graph in the order determined by the rotation system at each vertex. The order of traversal is the order of the edges in the disk frame representation of the ribbon graph.

In [4]:
boundary = get_boundary_data(graph)
print("Edge sequence around the face:", boundary["edge_sequences"][0])
print("Vertex sequence around the face:", boundary["vertex_sequences"][0])
print("Sewn boundary positions for each edge:")
pprint(boundary["sewing"])

assert len(boundary["edge_sequences"][0]) == 2 * len(edges)
assert all(len(positions) == 2 for positions in boundary["sewing"].values())

Edge sequence around the face: (0, 1, 2, 0, 1, 2)
Vertex sequence around the face: (0, 1, 0, 1, 0, 1)
Sewn boundary positions for each edge:
{0: ((0, 0), (0, 3)), 1: ((0, 1), (0, 4)), 2: ((0, 2), (0, 5))}


## Adding metric and discretization data

The ribbon graph in Kontsevich's formulation of the moduli space of Riemann surfaces is assigned a positive length $\ell_a$ for each graph edge. In the discretization scheme, this length is an integer, corresponding to the number of discretized lattice sites in the given edge segment. Each edge occurs twice around the disk boundary, so the total boundary length is $2\sum_a\ell_a$. We denote the total boundary length as $2L$.

In [5]:
# choose positive integer lengths for each edge segment. There are six total edge segments on the boundary of the disc, 
# but each is sewn together with another edge, so there are three indpendent edge segments.
edge_lengths = (8, 10, 12)
metric_boundary = get_boundary_data(graph, edge_lengths=edge_lengths)

print("Edge lengths:", metric_boundary["edge_lengths"])
print("Boundary-segment starting sites:", metric_boundary["boundary_segment_starts"][0])
print("Total disk-boundary length:", metric_boundary["boundary_lengths"][0])
# Note that although there are three independent edge segments, the total circumference of the disc
# is not a moduli parameter, and therefore one of the lengths is redundant.

assert metric_boundary["boundary_lengths"][0] == 2 * sum(edge_lengths)

Edge lengths: (8, 10, 12)
Boundary-segment starting sites: (0, 8, 18, 30, 38, 48)
Total disk-boundary length: 60


## Ribbon graph generation at genus 2

For genus 2 with one face, there are nine non-isomorphic ribbon graphs. Every ribbon graph has six vertices and nine edges.

In [6]:
graphs_g2 = generate_ribbon_graphs(genus=2, n_faces=1)
print(f"Number of genus-2 one-face topologies: {len(graphs_g2)}")

for index, candidate in enumerate(graphs_g2, start=1):
    candidate_boundary = get_boundary_data(candidate)
    print(
        f"topology {index}: "
        f"V={len(candidate[1])}, E={len(candidate[0])}, "
        f"boundary occurrences={len(candidate_boundary['edge_sequences'][0])}"
    )

assert len(graphs_g2) == 9

Number of genus-2 one-face topologies: 9
topology 1: V=6, E=9, boundary occurrences=18
topology 2: V=6, E=9, boundary occurrences=18
topology 3: V=6, E=9, boundary occurrences=18
topology 4: V=6, E=9, boundary occurrences=18
topology 5: V=6, E=9, boundary occurrences=18
topology 6: V=6, E=9, boundary occurrences=18
topology 7: V=6, E=9, boundary occurrences=18
topology 8: V=6, E=9, boundary occurrences=18
topology 9: V=6, E=9, boundary occurrences=18


## Non-exhaustive sampling at higher genus

For higher genus, there are a large number of ribbon graphs, so it can be computationally expensive to generate all ribbon graphs. `sample_ribbon_graph` instead implements a heuristic algorithm to search for one ribbon graph.

In [7]:
#there are 1,349,005 ribbon graphs of genus 4 with 1 face.
sampled_graph, metadata = sample_ribbon_graph(genus=4, n_faces=1, seed=7)
sampled_boundary = get_boundary_data(sampled_graph)

print(f"Sampled graph: V={len(sampled_graph[1])}, E={len(sampled_graph[0])}")
print("Sampling metadata:")
pprint(metadata)
print("Number of face boundaries:", sampled_boundary["n_faces"])

repeated_graph, repeated_metadata = sample_ribbon_graph(genus=4, n_faces=1, seed=7)
assert sampled_graph == repeated_graph
assert metadata == repeated_metadata

Sampled graph: V=14, E=21
Sampling metadata:
{'graph_seed': 647892279,
 'graph_trial': 0,
 'orientation_mask': 13455,
 'rotation_trial': 3,
 'topology_seed': 7}
Number of face boundaries: 1
